In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [5]:
%%file consumer_anomalie.py
from kafka import KafkaConsumer
import json
from datetime import datetime
from collections import defaultdict

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

history = defaultdict(list)

for message in consumer:
    tx = message.value
    u_id = tx['user_id']
    now = datetime.fromisoformat(tx['timestamp'])
    history[u_id].append(now)
    history[u_id] = [t for t in history[u_id] if (now - t).total_seconds() <= 60]
    if len(history[u_id]) > 3:
        print(f"ALERT: Użytkownik {u_id} wykonał {len(history[u_id])} transakcje w ciągu 60s!")
        print(f"Ostatnia: {tx['tx_id']} w {tx['store']}")

Overwriting consumer_anomalie.py
